# Refusal Ablation — Demo

Orchestration notebook for the `refusal-ablation` repo. All logic lives in `src/` — this notebook just wires it together for a given config, so results are reproducible from code, not from notebook-only cells.


## Setup: clone the repo (for Colab) and install deps

In [ ]:
!git clone -q https://github.com/FarrahTharwat/refusal-ablation.git
%cd refusal-ablation
!pip install -q datasets pyyaml


## Load config and model

In [ ]:
import yaml
import torch

from src.data import build_splits
from src.metrics import filter_refused, refusal_rate
from src.model_utils import load_model
from src.extract_direction import select_layer, extract_direction
from src.sweep import run_alpha_sweep, run_beta_sweep, save_results

CONFIG_PATH = "configs/qwen1.5-1.8b.yaml"   # swap this to try a different model
cfg = yaml.safe_load(open(CONFIG_PATH))
model_alias = cfg["model_name"].split("/")[-1]

torch.manual_seed(cfg["seed"])
model, tok = load_model(cfg)
print(f"{cfg['model_name']} loaded, {len(model.model.layers)} layers")


## Build data splits (extraction set + disjoint held-out eval set)

In [ ]:
harmful_extract_pool, harmful_eval_pool, harmless_extract_pool, harmless_eval_pool = build_splits(cfg)

harmful_extract = filter_refused(model, tok, harmful_extract_pool)
harmless_extract = harmless_extract_pool[:len(harmful_extract)]
print(f"{len(harmful_extract)} harmful extraction prompts survived the refusal filter")

harmful_eval = filter_refused(model, tok, harmful_eval_pool)[: cfg["eval_n"]]
harmless_eval = harmless_eval_pool[: len(harmful_eval)]
print(f"held-out eval sets: {len(harmful_eval)} harmful, {len(harmless_eval)} harmless")


## Select the best layer, then extract the final direction

In [ ]:
best_layer, layer_scores = select_layer(model, tok, cfg, harmful_extract, harmless_extract, harmful_eval)
direction = extract_direction(model, tok, harmful_extract, harmless_extract, best_layer)
ADD_LAYER = best_layer


## Baseline sanity check

In [ ]:
baseline_harmful_rr = refusal_rate(model, tok, harmful_eval)
baseline_harmless_rr = refusal_rate(model, tok, harmless_eval)
print(f"baseline harmful refusal rate:  {baseline_harmful_rr:.2f}")
print(f"baseline harmless refusal rate: {baseline_harmless_rr:.2f}")


## Alpha/beta sweep — the headline artifact

In [ ]:
alpha_df = run_alpha_sweep(model, tok, direction, ADD_LAYER, harmful_eval, harmless_eval, cfg["alphas"])
beta_df = run_beta_sweep(model, tok, direction, ADD_LAYER, harmful_eval, harmless_eval, cfg["betas"])
save_results(alpha_df, beta_df, results_dir="results", model_alias=model_alias)


## Notes

- To try a different model, change `CONFIG_PATH` above to point at another file in `configs/` and rerun — nothing else in this notebook needs to change.
- `results/` will contain `alpha_sweep_<model>.csv`, `beta_sweep_<model>.csv`, and `crossover_<model>.png` after this runs — commit these for the blog post.
- I could not execute this myself (no GPU / no Hugging Face Hub access in my sandbox) — please sanity-check as you run it in Colab.
